In [1]:
from fastai.vision.all import *
import pandas as pd
import json
from functools import partial
import os



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
path = Path("/kaggle/input/cassava-leaf-disease-classification/")

_ = path  # keep variable alive; no side effects



In [3]:
train_df = pd.read_csv(path / "train.csv")
with open(path / "label_num_to_disease_map.json") as f:
    label_dict = json.load(f)
label_dict = {int(k): v for k, v in label_dict.items()}

df = train_df.set_index("image_id")
labels = df["label"].to_dict()


def get_label(labels, x):
    return labels[x.name]


def _build_is_valid_map(_train_df, seed=42, frac=0.2):
    _df = _train_df.copy()
    set_seed(seed, reproducible=True)
    valid_df = _df.groupby("label", group_keys=False).sample(
        frac=frac, random_state=seed
    )
    valid_set = set(valid_df["image_id"].values.tolist())
    return {img_id: (img_id in valid_set) for img_id in _df["image_id"].values.tolist()}


_IS_VALID_MAP = _build_is_valid_map(train_df, seed=42, frac=0.2)

_TRAIN_FILES = None


def get_data(labels, bs=64, presize=500, resize=384):
    tfms = [
        Rotate(90),
        Warp(magnitude=0.4, p=1.0),
        Zoom(min_zoom=0.9, max_zoom=1.3),
        Brightness(max_lighting=0.5),
        Flip(),
        Contrast(),
        Resize(resize),
    ]
    comp = setup_aug_tfms(tfms)

    is_valid_map = _IS_VALID_MAP

    def _get_items(p):
        global _TRAIN_FILES
        if _TRAIN_FILES is None:
            _TRAIN_FILES = get_image_files(p)
        return _TRAIN_FILES

    def _splitter(items):
        mget = is_valid_map.get
        train, valid = [], []
        for i, o in enumerate(items):
            if mget(o.name, False):
                valid.append(i)
            else:
                train.append(i)
        return train, valid

    ncpu = os.cpu_count() or 2
    nw = min(8, max(2, ncpu // 2))

    dl_kwargs = dict(num_workers=nw, pin_memory=True)
    try:
        dl_kwargs.update(dict(persistent_workers=True, prefetch_factor=4))
    except Exception:
        pass

    dls = DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=_get_items,
        get_y=partial(get_label, labels),
        splitter=_splitter,
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)],
    ).dataloaders(path / "train_images", bs=bs, **dl_kwargs)

    return dls




In [4]:
set_seed(42, reproducible=True)

torch.backends.cudnn.benchmark = True

dls = get_data(labels, bs=128, presize=384, resize=384)
learn = cnn_learner(dls, models.resnet50, metrics=[accuracy], pretrained=True)

sample_sub = pd.read_csv(path / "sample_submission.csv")
sub_ids = sample_sub["image_id"].tolist()

test_files = [(path / "test_images" / fn) for fn in sub_ids]

missing = [p.name for p in test_files if not p.exists()]
if len(missing) > 0:
    raise FileNotFoundError(
        f"Missing {len(missing)} test images (first 5): {missing[:5]}"
    )

# BUGFIX: build the test dataloader from the *explicit ordered list* of test_files.
# Using DataBlock.dataloaders(path) can re-scan disk and change ordering/count, causing mismatch vs sample_submission.
ncpu = os.cpu_count() or 2
nw = min(8, max(2, ncpu // 2))
test_dl = learn.dls.test_dl(
    test_files,
    bs=128,
    num_workers=nw,
    pin_memory=True,
    shuffle=False,
)

ext_ckpt_dir = Path("/kaggle/input/resnet50/models")

learn.model.eval()
device = learn.dls.device


def _predict_folds_avg_singlepass(learn, dl, ckpt_paths, n_classes=5):
    mdl = learn.model
    dev = learn.dls.device

    use_cuda = torch.cuda.is_available() and (
        str(dev).startswith("cuda") or getattr(dev, "type", "") == "cuda"
    )
    if use_cuda:
        mdl = mdl.to(memory_format=torch.channels_last)

    n_items = len(dl.dataset)
    sum_preds = torch.zeros((n_items, n_classes), dtype=torch.float32, device=dev)

    mdl_fwd = mdl.__call__
    float32 = torch.float32

    existing = [p for p in ckpt_paths if p.exists()]
    n_folds_loaded = 0

    with torch.inference_mode():
        for ckpt in existing:
            # BUGFIX: fastai Learner.load expects a file name/path *without* '.pth' (it appends .pth).
            # Passing ckpt.with_suffix('') can be wrong if suffix isn't exactly '.pth' or path handling differs.
            # Use the standard "stem" with parent by providing the full path sans suffix via ckpt.parent/ckpt.stem.
            learn.load(ckpt.parent / ckpt.stem)
            mdl.eval()
            n_folds_loaded += 1

            start = 0
            for b in dl:
                xb = b[0] if isinstance(b, (tuple, list)) else b
                bs = xb.shape[0]

                xb = xb.to(dev, non_blocking=True)
                if use_cuda:
                    xb = xb.contiguous(memory_format=torch.channels_last)

                out = mdl_fwd(xb).to(dtype=float32)
                sum_preds[start : start + bs].add_(out)
                start += bs

    if n_folds_loaded == 0:
        return None

    sum_preds.div_(n_folds_loaded)
    return sum_preds.detach().cpu()


def _predict_single(learn, dl, n_classes=5):
    mdl = learn.model
    dev = learn.dls.device
    use_cuda = torch.cuda.is_available() and (
        str(dev).startswith("cuda") or getattr(dev, "type", "") == "cuda"
    )
    if use_cuda:
        mdl = mdl.to(memory_format=torch.channels_last)

    n_items = len(dl.dataset)
    preds_dev = torch.zeros((n_items, n_classes), dtype=torch.float32, device=dev)
    mdl_fwd = mdl.__call__

    with torch.inference_mode():
        start = 0
        for b in dl:
            xb = b[0] if isinstance(b, (tuple, list)) else b
            bs = xb.shape[0]
            xb = xb.to(dev, non_blocking=True)
            if use_cuda:
                xb = xb.contiguous(memory_format=torch.channels_last)
            out = mdl_fwd(xb).to(dtype=torch.float32)
            preds_dev[start : start + bs].copy_(out)
            start += bs

    return preds_dev.detach().cpu()


_TTA_N = 6  # kept for compatibility with original variables; not used

ckpt_paths = [ext_ckpt_dir / f"resnet50-full-fold_{fold}.pth" for fold in range(3)]
avg_preds = _predict_folds_avg_singlepass(learn, test_dl, ckpt_paths, n_classes=5)

if avg_preds is None:
    learn.fine_tune(5, base_lr=3e-3)
    avg_preds = _predict_single(learn, test_dl, n_classes=5)

# Sanity check: prediction count must match submission rows
if avg_preds is None or len(avg_preds) != len(sample_sub):
    raise RuntimeError(
        f"Prediction length mismatch: preds={None if avg_preds is None else len(avg_preds)} vs submission={len(sample_sub)}"
    )



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,1.184177,0.780813,0.748397,01:25


epoch,train_loss,valid_loss,accuracy,time
0,0.681685,0.556451,0.824786,01:36
1,0.534976,0.455857,0.851763,01:34
2,0.446804,0.400440,0.867788,01:34
3,0.399261,0.382002,0.873932,01:34
4,0.365812,0.377957,0.875801,01:33


In [5]:
pred_labels = avg_preds.argmax(dim=1).cpu().numpy().astype(int)

# test_files were constructed in sample_submission order, and test_dl preserves that order.
ordered_labels = pred_labels
assert len(ordered_labels) == len(sample_sub)



In [6]:
submission = pd.DataFrame(
    {"image_id": sample_sub["image_id"].values, "label": ordered_labels}
)
submission = submission[["image_id", "label"]]
assert len(submission) == len(sample_sub)
submission.to_csv("submission.csv", index=False)

submission.head()

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,1
3,1234555380.jpg,1
4,1234571117.jpg,3
